In [102]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [103]:
import os
from pathlib import Path
from pprint import pprint
import sys
from typing import Optional, List, Dict, Any, Tuple
if '..' not in sys.path: sys.path.append('..')

from dataclasses import dataclass
from datasets import load_dataset
from datasets.arrow_dataset import Dataset
import numpy as np
from matplotlib import pyplot as plt
from pydantic_yaml import parse_yaml_file_as
import torch
from torch import nn
import torch.nn.functional as F
from transformers import AutoTokenizer, PreTrainedTokenizer

from mllm.data.wiki.dswiki import WikiDsLoader
from mllm.data.utils import RandomInputTokenizer, RandomInputTokenizerV2, TokensSubset, TokensSubsetV2, tokens_subsets_to_tensors, tokens_subsets_v2_to_tensors
from mllm.exp.args import MIXED_DECODER_MODEL_CFG_FNAME
from mllm.model.mixed_decoder import MixedDecoder
from mllm.config.model import MixedDecoderCfg
from mllm.train.encdec_graph_bert import MaskedCiteDataset, create_masked_cite_dataloader, load_split_wiki_dataset

In [104]:
# DATA_PATH = Path(os.path.expandvars('$HOME')) / 'data'
DATA_PATH = Path('Q:/data')
# WIKI_DS_NAME = '20200501.en'
WIKI_DS_NAME = '20220301.en'

TRAIN_ENCDEC_BERT_PATH = DATA_PATH / 'train_mllm_encdec_bert'
mixed_decoder_subdir = 'mixeddecoder-20260304_105309-pre_encdecbert20260110193915-bertbaseuncased-d768-embEncCls-inp128-decGpt2-decmgpt2-msl384-sepT-pallF-eer4-ewn10x10-frzencF-trn_lr5e-05_bs30'

mixed_decoder_train_path = TRAIN_ENCDEC_BERT_PATH / mixed_decoder_subdir
mixed_decoder_snapshot_fpath = mixed_decoder_train_path / 'best.pth'
mixed_decoder_model_cfg_fpath = mixed_decoder_train_path / MIXED_DECODER_MODEL_CFG_FNAME

device_name = 'cpu'
# device_name = 'cuda'

device = torch.device(device_name)
print(device)

cpu


## Wikipedia dataset loading

In [105]:
# dss = load_dataset('wikipedia', WIKI_DS_NAME, beam_runner='DirectRunner', cache_dir=str(DATA_PATH))
dss = load_dataset('wikipedia', WIKI_DS_NAME, cache_dir=str(DATA_PATH), trust_remote_code=True)
ds: Dataset = dss['train']
n_docs = len(ds)
print(f'Wikipedia {WIKI_DS_NAME} docs: {n_docs}')

Loading dataset shards:   0%|          | 0/41 [00:00<?, ?it/s]

Wikipedia 20220301.en docs: 6458670


### Masked dataset loading

In [111]:
model_cfg = parse_yaml_file_as(MixedDecoderCfg, mixed_decoder_model_cfg_fpath)
pprint(model_cfg.dict())

tkz = AutoTokenizer.from_pretrained(model_cfg.enc_bert.pretrained_model_name)
inp_len = model_cfg.enc_bert.inp_len
n_special_toks = 3
prompt_all = model_cfg.prompt_all
mask_cfg = None
ds_cite = MaskedCiteDataset(ds, tkz, max_seq_len=inp_len, n_special_toks=n_special_toks, mask_cfg=mask_cfg, prompt_all=prompt_all, device=device)

{'d_model': 768,
 'decoder_model_name': 'gpt2',
 'decoder_type': <MixedDecoderType.Gpt2: 'gpt2'>,
 'emb_exp_rate': 4,
 'emb_win_max_size': 10,
 'emb_win_min_size': 10,
 'enc_bert': {'d_model': 768,
              'emb2_tok_name': '',
              'emb_type': <BertEmbType.Cls: 'cls'>,
              'inp_len': 128,
              'pad_token_id': 0,
              'pretrained_model_name': 'bert-base-uncased',
              'tokenizer_name': 'bert-base-uncased'},
 'max_seq_len': 384,
 'prompt_all': False,
 'train_cfg': {'batch_size': 30,
               'freeze_encoder': False,
               'learning_rate': 5e-05,
               'learning_rate_scheduler': {'cls_name': 'ReduceLROnPlateau',
                                           'module_path': 'torch.optim.lr_scheduler',
                                           'params': {'factor': 0.5,
                                                      'min_lr': 1e-08,
                                                      'mode': 'min',
            

### Inspect a masked dataset batch samples

In [112]:
batch_off = 10
batch_size = 5
inds = np.arange(batch_off, batch_off + batch_size)
inds = inds.tolist()
batch = ds_cite.get_batch(inds)

Token indices sequence length is longer than the specified maximum sequence length for this model (9432 > 512). Running this sequence through the model will result in indexing errors


In [113]:
print(tkz)

BertTokenizerFast(name_or_path='bert-base-uncased', vocab_size=30522, model_max_length=512, is_fast=True, padding_side='right', truncation_side='right', special_tokens={'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, clean_up_tokenization_spaces=False, added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	100: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	101: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	102: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	103: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
}
)


In [114]:
print(f'inp_toks: {batch.inp_toks.shape}. inp_attn_mask: {batch.inp_att_mask.shape}. first: {batch.inp_toks[0, 0]}. last: {batch.inp_toks[0, -1]}.')
print(f'prompts_toks: {batch.prompts_toks.shape}. prompts_attn_mask: {batch.prompts_att_mask.shape}. first: {batch.prompts_toks[0, 0]}. last: {batch.prompts_toks[0, -1]}.')
print(f'cites_toks: {batch.cites_toks.shape}. cites_attn_mask: {batch.cites_att_mask.shape}.')

inp_toks: torch.Size([5, 128]). inp_attn_mask: torch.Size([5, 128]). first: 101. last: 102.
prompts_toks: torch.Size([5, 29]). prompts_attn_mask: torch.Size([5, 29]). first: 101. last: 102.
cites_toks: torch.Size([5, 117]). cites_attn_mask: torch.Size([5, 117]).


In [115]:
print('inp_toks[0]:', tkz.decode(batch.inp_toks[0].cpu().numpy()))
print('prompts_toks[0]:', tkz.decode(batch.prompts_toks[0].cpu().numpy()))
print('cites_toks[0]:', tkz.decode(batch.cites_toks[0].cpu().numpy()))

inp_toks[0]: [CLS] of the new rule made no direct mention of that film. the best international feature film award does not require a u. s. release. it requires the film to be submitted as its country ' s official selection. the best winkdah perth documentary feature award requires either week - long releases in both los angeles county and new york city during the previous calendar year, or a qualifying award at a competitive film festival from the documentary feature qualifying festival list ( regardless of any public exhibition or distribution ), or submission in the international feature film category as its country ' goddamn refugee influenced s official selection. the qualifying theatrical runs must meet the same requirements [SEP]
prompts_toks[0]: [CLS] cite tag begin : " winkdah perth ". cite tag end : " goddamn refugee influenced ". produce output text between these tags. [SEP]
cites_toks[0]: documentary feature award requires either week - long releases in both los angeles coun

## Model loading and inference

In [116]:
model_cfg.emb_win_min_size = model_cfg.emb_win_max_size

chkpt = torch.load(mixed_decoder_snapshot_fpath, map_location=device)
model = MixedDecoder(model_cfg, tkz).to(device)
model.load_pretrained(chkpt)
del chkpt
model.eval()
None

Load 351


### Run forward pass (with windowing disabled for inference)

In [117]:

batch_off = 10
# batch_size = 4 * model_cfg.emb_win_max_size
batch_size = 15
inds = np.arange(batch_size) + batch_off
inds = inds.tolist()
batch = ds_cite.get_batch(inds)

In [118]:
with torch.no_grad():
    loss_dict, dec_logits = model.run_on_text_citation(batch)
print(loss_dict)

{'loss': tensor(0.7688)}


In [119]:
logits = dec_logits.detach().numpy()
logits.shape

(10, 172, 50257)

In [120]:
probs_pred = torch.softmax(dec_logits, dim=-1)
print('probs shape:', probs_pred.shape)
docs_toks_out = torch.argmax(probs_pred, dim=-1)
docs_toks_out = docs_toks_out.detach().numpy()
print('toks_out shape:', docs_toks_out.shape)

probs shape: torch.Size([10, 172, 50257])
toks_out shape: (10, 172)


In [121]:
n_ctx = model_cfg.emb_win_max_size
if model.cfg.emb_exp_rate > 0:
    n_ctx = n_ctx * model.cfg.emb_exp_rate
prefix_len = n_ctx
if model.cfg.use_sep:
    prefix_len += 1
prefix_len += batch.prompts_toks.shape[1]
target_start_idx = prefix_len - 1

if model.cfg.prompt_all:
    target_len = batch.inp_toks.shape[1] - 1  # strip CLS
else:
    target_len = batch.cites_toks.shape[1]

print(f'n_ctx: {n_ctx}, prefix_len: {prefix_len}, target_start_idx: {target_start_idx}, target_len: {target_len}')

# Extract predicted tokens for the target region
target_logits = dec_logits[:, target_start_idx:target_start_idx + target_len, :]
target_toks_pred = torch.argmax(target_logits, dim=-1).detach().numpy()
print('target_toks_pred shape:', target_toks_pred.shape)

n_ctx: 40, prefix_len: 70, target_start_idx: 69, target_len: 103
target_toks_pred shape: (10, 103)


In [128]:
for i in range(batch_size):
    print(f'=== Sample {i} ===')
    if model.cfg.prompt_all:
        gt_toks = batch.inp_toks[i, 1:]  # strip CLS
    else:
        gt_toks = batch.cites_toks[i]
    print('ground truth:')
    print(tkz.decode(gt_toks.cpu().numpy()))
    print('predicted:')
    print(tkz.decode(target_toks_pred[i]))
    print()

=== Sample 0 ===
ground truth:
reason for the move to late february and early march is also to avoid the awards ceremony occurring so close to the religious holidays of passover and easter, which for decades had been a grievance from members and the general public. advertising is somewhat restricted, however, as traditionally no movie studios or competitors of official academy award sponsors may advertise during the telecast. the production of the academy awards telecast currently holds the distinction of winning the most emmys in history, with 47 wins and 195 nominations overall since that award
predicted:
reason for their move to late february and early february are also ongoing conclude the awards ceremony postponed so close to the gathering view of passover, easter, which had decades had been a grievance between general and general general public. this is not restricted, however, as well no restaurants venues or hosts of official productions shows announcements may bevertise during

## Encoder embeddings inspection

In [123]:
with torch.no_grad():
    inp_enc_embs = model.run_enc(batch.inp_toks, batch.inp_att_mask)
print('inp_enc_embs shape:', inp_enc_embs.shape)

inp_embs = inp_enc_embs.cpu().detach().numpy()
print('min:', inp_embs.min(), 'max:', inp_embs.max(), 'mean:', inp_embs.mean(), 'std:', inp_embs.std())

inp_enc_embs shape: torch.Size([10, 768])
min: -4.3990536 max: 4.1376295 mean: -0.021059195 std: 1.1592766


## Autoregressive generation from encoder embeddings

In [136]:
@torch.no_grad()
def generate_from_embs(
    model: MixedDecoder, inp_enc_embs: torch.Tensor,
    prompt_toks: torch.Tensor, prompt_att_mask: torch.Tensor,
    max_new_tokens: int = 100, temperature: float = 1.0,
    win_size: Optional[int] = None,
) -> torch.Tensor:
    """Autoregressive generation: given encoder CLS embeddings and prompt tokens,
    generate tokens one by one using the MixedDecoder.

    Args:
        model: MixedDecoder model in eval mode
        inp_enc_embs: (batch_size, d_model) - CLS embeddings from encoder
        prompt_toks: (batch_size, prompt_len) - prompt token ids
        prompt_att_mask: (batch_size, prompt_len) - prompt attention mask
        max_new_tokens: maximum number of tokens to generate
        temperature: sampling temperature (1.0 = greedy with argmax)
        win_size: size of embedding window per sample (None = use all batch embeddings)

    Returns:
        generated_toks: (batch_size, max_new_tokens) - generated token ids
    """
    batch_size = inp_enc_embs.shape[0]
    device = inp_enc_embs.device
    cfg = model.cfg

    # Determine embedding window indices
    win_indices = None
    if win_size is not None and win_size < batch_size:
        offset_before = 0  # sample's own embedding at position 0
        sample_idx = torch.arange(batch_size, device=device)
        j_idx = torch.arange(win_size, device=device)
        # win_indices[i, j] = (i - offset_before + j) % batch_size
        win_indices = (sample_idx.unsqueeze(1) - offset_before + j_idx.unsqueeze(0)) % batch_size

    # Build context embeddings
    if cfg.emb_exp_rate > 0:
        exp_embs = model.emb_exp(inp_enc_embs)
        exp_embs = exp_embs.view(batch_size, cfg.emb_exp_rate, model.d_dec)
        if win_indices is not None:
            # (batch_size, win_size, emb_exp_rate, d_dec) -> (batch_size, win_size * emb_exp_rate, d_dec)
            ctx_embs = exp_embs[win_indices]
            ctx_embs = ctx_embs.reshape(batch_size, win_indices.shape[1] * cfg.emb_exp_rate, model.d_dec)
        else:
            exp_embs = exp_embs.reshape(1, batch_size * cfg.emb_exp_rate, model.d_dec)
            ctx_embs = exp_embs.expand(batch_size, -1, -1)
    else:
        if win_indices is not None:
            ctx_embs = inp_enc_embs[win_indices]
        else:
            ctx_embs = inp_enc_embs.unsqueeze(0).expand(batch_size, -1, -1)

    # Project if needed
    if model.enc_proj is not None:
        ctx_embs = model.enc_proj(ctx_embs)

    n_ctx = ctx_embs.shape[1]

    # Build initial prefix: [ctx_embs, (sep), prompt_embs]
    parts_embs = [ctx_embs]
    parts_mask = [torch.ones((batch_size, n_ctx), dtype=torch.long, device=device)]

    if cfg.use_sep:
        sep_tok = torch.full((batch_size, 1), model.sep_token_id, dtype=torch.long, device=device)
        sep_emb = model.word_embeddings(sep_tok)
        parts_embs.append(sep_emb)
        parts_mask.append(torch.ones((batch_size, 1), dtype=torch.long, device=device))

    prompt_embs = model.word_embeddings(prompt_toks)
    parts_embs.append(prompt_embs)
    parts_mask.append(prompt_att_mask)

    input_embs = torch.cat(parts_embs, dim=1)  # (batch_size, prefix_len, d_dec)
    attention_mask = torch.cat(parts_mask, dim=1)

    generated_ids = []
    for step in range(max_new_tokens):
        total_len = input_embs.shape[1]
        pos_ids = torch.arange(total_len, device=device).unsqueeze(0)
        pos_embs = model.pos_emb(pos_ids)
        embs_with_pos = input_embs + pos_embs

        logits = model.run_decoder(embs_with_pos, attention_mask)
        next_logits = logits[:, -1, :]  # (batch_size, vocab_size)

        if temperature <= 0:
            next_tok = torch.argmax(next_logits, dim=-1)  # (batch_size,)
        else:
            probs = torch.softmax(next_logits / temperature, dim=-1)
            next_tok = torch.multinomial(probs, num_samples=1).squeeze(-1)

        generated_ids.append(next_tok)

        # Append new token embedding
        next_emb = model.word_embeddings(next_tok.unsqueeze(1))  # (batch_size, 1, d_dec)
        input_embs = torch.cat([input_embs, next_emb], dim=1)
        # Technically this isn't correct in case prompts have different lengths, but in our setting all prompts have the same length
        attention_mask = torch.cat([attention_mask, torch.ones((batch_size, 1), dtype=torch.long, device=device)], dim=1)

    return torch.stack(generated_ids, dim=1)  # (batch_size, max_new_tokens)

In [140]:
# batch_off = 10
# batch_size = 15
batch_off = 25
batch_size = 10
inds = np.arange(batch_size) + batch_off
inds = inds.tolist()
batch = ds_cite.get_batch(inds)

inp_enc_embs = model.run_enc(batch.inp_toks, batch.inp_att_mask)

In [141]:
max_new_tokens = 100
win_size = model_cfg.emb_win_max_size
gen_toks = generate_from_embs(
    model, inp_enc_embs,
    batch.prompts_toks, batch.prompts_att_mask,
    max_new_tokens=max_new_tokens, temperature=0,
    win_size=win_size,
)
print('gen_toks shape:', gen_toks.shape)

inp_toks = batch.inp_toks.cpu().numpy()
prompts_toks = batch.prompts_toks.cpu().numpy()
cites_toks = batch.cites_toks.cpu().numpy()
gen_toks_np = gen_toks.cpu().numpy()

for i in range(batch_size):
    print(f'=== Sample {i} ===')
    print('input:')
    print(tkz.decode(inp_toks[i]))
    print('prompt:')
    print(tkz.decode(prompts_toks[i]))
    print('ground truth:')
    print(tkz.decode(cites_toks[i]))
    print('generated:')
    print(tkz.decode(gen_toks_np[i]))
    print()

gen_toks shape: torch.Size([10, 100])
=== Sample 0 ===
input:
[CLS] in 2015. originally based on letterman lord arrows the english alphabet, ascii encodes 128 specified characters into seven - bit integers as shown by the ascii chart above. ninety - five of the encoded characters are printable : these include the digits 0 to 9, lowercase letters aphonic postgraduatedication to z, uppercase letters a to z, and punctuation symbols. in addition, the original ascii specification included 33 non - printing control codes which originated with teletype machines ; most of these are now obsolete, although a few are still commonly used, such as the carriage return, line feed, [SEP]
prompt:
[CLS] cite tag begin : " letterman lord arrows ". cite tag end : "phonic postgraduatedication ". produce output text between these tags. [SEP]
ground truth:
the english alphabet, ascii encodes 128 specified characters into seven - bit integers as shown by the ascii chart above. ninety - five of the encoded cha

## Decoding with noisy embeddings

In [ ]:
# Add small noise to encoder embeddings and compare generation quality
noise_std = 1e-2
noise = torch.randn_like(inp_enc_embs) * noise_std
inp_enc_embs_noisy = inp_enc_embs + noise

gen_toks_noisy = generate_from_embs(
    model, inp_enc_embs_noisy,
    batch.prompts_toks, batch.prompts_att_mask,
    max_new_tokens=max_new_tokens, temperature=0,
    win_size=win_size,
)

for i in range(batch_size):
    print(f'=== Sample {i} ===')
    print('input:')
    print(tkz.decode(batch.inp_toks[i].cpu().numpy()))
    print('generated (clean):')
    print(tkz.decode(gen_toks[i].cpu().numpy()))
    print('generated (noisy, std={:.0e}):'.format(noise_std))
    print(tkz.decode(gen_toks_noisy[i].cpu().numpy()))
    print()

=== Sample 0 ===
input:
[CLS] olympic games. another courtesy melted lone reason for the move to late february and early march is also to avoid the awards ceremony occurring so close to the religious holidays of passover and easter, which for decades had been a grievance from members and the general public. advertising is somewhat restricted, however, as traditionally no movie studios or competitors of official academy award sponsors may advertise during the telecast. the production of the academy awards telecast currently holds the distinction of winning the most emmys in history, with 47 wins and 195 nominations overall since that award magnetic compliant emanuel ' s own launch in 1949. after many years of being held [SEP]
generated (clean):
reason for their move to be late in the fifth and last months. the international symposium will be frequently prepared for views of the daytime to avoid straight and a compromise of members and fblies in the year. history the conference is diffic

In [131]:
win_size = model_cfg.emb_win_max_size // 2
offset_before = torch.randint(0, win_size, (1,)).item() if win_size > 1 else 0
sample_idx = torch.arange(batch_size, device=inp_enc_embs.device)
j_idx = torch.arange(win_size, device=inp_enc_embs.device)
print(sample_idx)
print(j_idx)
# win_indices[i, j] = (i - offset_before + j) % batch_size
# Ensures each sample's own embedding is always at position offset_before
win_indices = (sample_idx.unsqueeze(1) - offset_before + j_idx.unsqueeze(0)) % batch_size

tensor([0, 1, 2, 3, 4, 5, 6, 7, 8, 9])
tensor([0, 1, 2, 3, 4])


In [132]:
print(win_indices)

tensor([[9, 0, 1, 2, 3],
        [0, 1, 2, 3, 4],
        [1, 2, 3, 4, 5],
        [2, 3, 4, 5, 6],
        [3, 4, 5, 6, 7],
        [4, 5, 6, 7, 8],
        [5, 6, 7, 8, 9],
        [6, 7, 8, 9, 0],
        [7, 8, 9, 0, 1],
        [8, 9, 0, 1, 2]])


In [133]:
print(offset_before)

1
